# 04 · Evaluation, Uncertainty & Decision Theory

**Owner: Member 4.**

Companion to §§9–10 of `main.ipynb`. We add learning-curve
analysis, threshold sensitivity, and per-subgroup metrics.

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
np.random.seed(42)

In [ ]:
from src.data_loader import load_heart_disease
from src.preprocessing import (
    PreprocessConfig, build_dataset, train_test_split_df, variable_state_names,
)
from src.expert_network import build_expert_dag
from src.parameter_learning import ParameterFitConfig, fit_parameters
from src.inference import predict_proba

df = build_dataset(load_heart_disease(), PreprocessConfig())
train, test = train_test_split_df(df, PreprocessConfig())
states = variable_state_names(df)
bn = fit_parameters(build_expert_dag(), train, ParameterFitConfig(method='bayes'), state_names=states)
positive = sorted(bn.get_cpds('target').state_names['target'])[-1]
bn_probs = predict_proba(bn, test, target='target')[positive].values
y_test = test['target'].astype(int).values

## Learning curve

How well does the Bayesian Network perform with only k% of
the training fold? This tells us how data-hungry the model is.

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss

fractions = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
rows = []
for frac in fractions:
    sub = train.sample(frac=frac, random_state=0)
    bn_k = fit_parameters(build_expert_dag(), sub, ParameterFitConfig(method='bayes'), state_names=states)
    probs = predict_proba(bn_k, test, target='target')[positive].values
    rows.append({
        'fraction': frac,
        'n_train': len(sub),
        'AUC': roc_auc_score(y_test, probs),
        'log_loss': log_loss(y_test, np.clip(probs, 1e-7, 1-1e-7)),
    })
lc = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lc['fraction'], lc['AUC'], 'o-'); axes[0].set_title('Test AUC'); axes[0].set_xlabel('train fraction')
axes[1].plot(lc['fraction'], lc['log_loss'], 'o-', color='crimson'); axes[1].set_title('Test log-loss')
plt.show()
lc

## Per-subgroup metrics

Group AUC by sex and age band — does the model perform
equitably?

In [ ]:
df_test = test.copy()
df_test['prob'] = bn_probs
df_test['y'] = y_test

rows = []
for (subgroup_var, subgroup_val), grp in df_test.groupby(['sex']):
    if grp['y'].nunique() < 2: continue
    rows.append({'subgroup': f'sex={subgroup_val}', 'n': len(grp),
                 'AUC': roc_auc_score(grp['y'], grp['prob'])})
for age_val, grp in df_test.groupby('age'):
    if grp['y'].nunique() < 2: continue
    rows.append({'subgroup': f'age={age_val}', 'n': len(grp),
                 'AUC': roc_auc_score(grp['y'], grp['prob'])})
pd.DataFrame(rows)

## Threshold sensitivity under different FN:FP cost ratios

In [ ]:
from src.uncertainty import UtilityMatrix, optimal_threshold

rows = []
for ratio in [1, 2, 5, 10, 20, 50]:
    util = UtilityMatrix(cost_fp=1.0, cost_fn=float(ratio))
    t, _ = optimal_threshold(y_test, bn_probs, utility=util)
    rows.append({'FN:FP ratio': ratio, 'optimal threshold': t})
pd.DataFrame(rows)

## Width of credible interval by confidence band

Where in the probability range is the BN most *uncertain*?

In [ ]:
from src.uncertainty import UncertaintyConfig, posterior_predictive

uq = posterior_predictive(build_expert_dag(), train, test, state_names=states,
                          cfg=UncertaintyConfig(n_posterior_samples=15))
widths = uq['ci_high'] - uq['ci_low']
bands = pd.cut(uq['mean'], bins=[0, 0.2, 0.4, 0.6, 0.8, 1.0])
pd.DataFrame({'mean_band': bands, 'CI_width': widths}).groupby('mean_band', observed=True)['CI_width'].agg(['mean', 'count'])